# Tables

All tables for the paper. Each is displayed here and written to `../tables/` as `.csv` and a booktabs `.tex` fragment.

Most results are read from the CSVs written by `data_correction_and_model_training.ipynb` (`pipelinedata/05_final/`). The free-slope circular (Möbius) baselines, and the leave-one-wind-speed-out baselines, are not in that notebook, so they are fit here with the same trajectory-grouped split (seed 42, 33% test) and the same evaluation protocol as the OLS baselines.

1. Linear / circular baselines and the ANN, by regime
2. Per-trajectory error distribution, all models
3. ANN error by regime
4. Leave-one-wind-speed-out (ANN and baselines)
5. Robustness across training seeds
6. Architecture and loss selection (grouped CV)
7. Ensemble-agreement calibration
8. ANN error vs. trajectory covariates

In [1]:
import os, sys
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
sys.path.insert(0, os.path.abspath('../utils'))
from utils import circular_distance
from IPython.display import display

D = '../pipelinedata/05_final/'
OUT = '../tables/'
os.makedirs(OUT, exist_ok=True)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
REGIMES = ['upwind', 'crosswind', 'downwind']
THRESH = 15.0

def classify_direction(a):
    deg = np.degrees(np.arctan2(np.sin(a), np.cos(a)))
    return np.where(np.abs(deg) <= 45, 'upwind', np.where(np.abs(np.abs(deg) - 180) <= 45, 'downwind', 'crosswind'))

def bootstrap_ci(v, n_boot=5000, seed=0):
    v = np.asarray(v); rng = np.random.default_rng(seed)
    med = np.median(rng.choice(v, size=(n_boot, len(v)), replace=True), axis=1)
    lo, hi = np.percentile(med, [2.5, 97.5])
    return float(np.median(v)), float(lo), float(hi)

def ci_str(m, lo, hi): return f'{m:.2f} [{lo:.2f}, {hi:.2f}]'

def esc(s): return str(s).replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

def save(df, name, caption, label, align=None):
    df.to_csv(f'{OUT}{name}.csv', index=False)
    align = align or 'l' + 'r' * (df.shape[1] - 1)
    lines = [r'\begin{table}[h!]', r'    \centering', f'    \\caption{{{caption}}}', f'    \\label{{tab:{label}}}',
             f'    \\begin{{tabular}}{{@{{}}{align}@{{}}}}', r'        \toprule',
             '        ' + ' & '.join(esc(c) for c in df.columns) + r' \\', r'        \midrule']
    for _, r in df.iterrows():
        lines.append('        ' + ' & '.join(esc(x) for x in r.values) + r' \\')
    lines += [r'        \bottomrule', r'    \end{tabular}', r'\end{table}']
    open(f'{OUT}{name}.tex', 'w').write('\n'.join(lines) + '\n')
    display(df)

## Setup: split, and linear / circular / Möbius baselines

Baselines follow the notebook's definitions: *linear* OLS regresses the heading unit vector on the Cartesian components of each kinematic vector; *circular* OLS regresses it on the unit direction only (degree-1 trigonometric polynomial). The additional **Möbius** model is the free-slope circular-circular regression from `failure_of_linear_models.ipynb`: $\hat\theta = \mu + 2\arctan(\beta\tan\frac{\theta_x-\nu}{2})$, fit by minimising $\sum(1-\cos e)$. It takes a single input angle, so it is fit once per kinematic vector.

In [2]:
data = pd.read_csv(D + 'all_wind_heading_and_trajectories_augmented_corrected_filtered_headx_heady.csv')
ids = data['trajec_objid'].unique()
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42).split(np.zeros((len(ids), 1)), groups=ids))
train_df = data[data['trajec_objid'].isin(set(ids[tr]))].reset_index(drop=True)
test_df  = data[data['trajec_objid'].isin(set(ids[te]))].reset_index(drop=True)
print(len(train_df), 'train frames,', len(test_df), 'test frames')

QUANT = {'thrust': ('thrust', 'thrust_angle'), 'air': ('airspeed', 'airspeed_angle'), 'ground': ('groundspeed', 'groundspeed_angle')}

def design(d, combo, kind):
    cols = [np.ones(len(d))]
    for q in combo:
        m, a = QUANT[q]
        if kind == 'linear': cols += [d[m].values * np.cos(d[a].values), d[m].values * np.sin(d[a].values)]
        else:                cols += [np.cos(d[a].values), np.sin(d[a].values)]
    return np.column_stack(cols)

def ols_angle(train, test, combo, kind):
    Xtr, Xte = design(train, combo, kind), design(test, combo, kind)
    coef = np.linalg.lstsq(Xtr, train[['heading_angle_x', 'heading_angle_y']].values, rcond=None)[0]
    p = Xte @ coef
    return np.arctan2(p[:, 1], p[:, 0])

def circ_mean(a): return np.arctan2(np.sin(a).mean(), np.cos(a).mean())
def mobius(x, mu, nu, lb): return mu + 2 * np.arctan(np.exp(lb) * np.tan((x - nu) / 2))

def fit_mobius(x, y):
    best = None
    for nu0 in np.linspace(0, 2 * np.pi, 8, endpoint=False):
        for lb0 in (-1.0, 0.0, 1.0, 2.0):
            r = minimize(lambda p: np.mean(1 - np.cos(y - mobius(x, *p))), [circ_mean(y - x), nu0, lb0],
                         method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-10, maxiter=4000))
            if best is None or r.fun < best.fun: best = r
    return best.x

def mobius_angle(train, test, q):
    a = QUANT[q][1]
    return mobius(test[a].values, *fit_mobius(train[a].values, train['heading_angle'].values))

def frame_errors(test, pred_angle):
    err = np.degrees(circular_distance(test['heading_angle'].values, pred_angle))
    return pd.DataFrame({'trajec_objid': test['trajec_objid'].values, 'err': err,
                         'regime': classify_direction(test['heading_angle'].values)})

def summarize(fdf):
    """per-trajectory medians (pooled and per regime), as in the paper"""
    pooled = fdf.groupby('trajec_objid')['err'].median()
    by_reg = fdf.groupby(['regime', 'trajec_objid'])['err'].median().reset_index()
    out = {'overall': pooled.median()}
    for r in REGIMES: out[r] = by_reg.loc[by_reg['regime'] == r, 'err'].median()
    return out, pooled, by_reg

# Möbius baselines on the paper's split
mobius_rows, mobius_traj = [], {}
for q in QUANT:
    s, pooled, _ = summarize(frame_errors(test_df, mobius_angle(train_df, test_df, q)))
    name = f'mobius: {q}'
    mobius_rows.append({'model': name, **s}); mobius_traj[name] = pooled
print(pd.DataFrame(mobius_rows))

# sanity check: reproduce one saved OLS baseline
ref = pd.read_csv(D + 'ols_baselines_by_regime.csv').set_index('model').loc['linear: thrust+air', 'overall']
chk, _, _ = summarize(frame_errors(test_df, ols_angle(train_df, test_df, ('thrust', 'air'), 'linear')))
print(f"check linear: thrust+air overall: saved {ref:.3f}, recomputed {chk['overall']:.3f}")

76304 train frames, 37434 test frames


            model    overall     upwind  crosswind   downwind
0  mobius: thrust  19.403505  67.390286  49.161461  12.637525
1     mobius: air   8.518929  31.974910  26.415106   6.549697
2  mobius: ground  17.035111  47.325518  47.569431   8.836963
check linear: thrust+air overall: saved 9.915, recomputed 9.915


## Table 1 — Linear, circular and Möbius baselines vs. the ANN, by flight regime

Median per-trajectory angular error (deg) on the held-out test set. Regimes are classified from the true heading relative to the wind axis (upwind $|\theta|\le45^\circ$, downwind within $45^\circ$ of $180^\circ$, otherwise crosswind).

In [3]:
ann_frames = pd.read_csv(D + 'ann_by_regime.csv')
ann_s, ann_pooled, ann_by_reg = summarize(ann_frames)
base = pd.read_csv(D + 'ols_baselines_by_regime.csv')
t1 = pd.concat([base, pd.DataFrame(mobius_rows), pd.DataFrame([{'model': 'ANN', **ann_s}])], ignore_index=True)
t1['family'] = t1['model'].str.split(':').str[0].map({'linear': 'Linear OLS', 'circular': 'Circular OLS',
                                                     'mobius': 'Möbius circular regression', 'ANN': 'ANN'})
t1['inputs'] = t1['model'].str.split(': ').str[-1].where(t1['model'] != 'ANN', 'groundspeed + airspeed')
t1 = t1.sort_values('overall', ascending=False).reset_index(drop=True)
table1 = t1[['family', 'inputs', 'overall', 'upwind', 'crosswind', 'downwind']].copy()
table1.columns = ['Model', 'Inputs', 'Overall', 'Upwind', 'Crosswind', 'Downwind']
best_idx = {c: t1[t1['family'] != 'ANN'][c.lower()].idxmin() for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']}
print('best non-ANN baseline per column:', {c: (t1.loc[i, 'model'], round(t1.loc[i, c.lower()], 2)) for c, i in best_idx.items()})
for c in ['Overall', 'Upwind', 'Crosswind', 'Downwind']: table1[c] = table1[c].map('{:.2f}'.format)
save(table1, 'table1_baselines_by_regime',
     'Median per-trajectory angular error (deg) of every linear, circular and M\\"obius baseline and of the ANN on the held-out test trajectories, overall and by flight regime. Sorted by overall error, worst first.',
     'baseline_regime', align='llrrrr')

best non-ANN baseline per column: {'Overall': ('mobius: air', np.float64(8.52)), 'Upwind': ('mobius: air', np.float64(31.97)), 'Crosswind': ('mobius: air', np.float64(26.42)), 'Downwind': ('linear: thrust+ground', np.float64(5.14))}


,Model,Inputs,Overall,Upwind,Crosswind,Downwind
0,Möbius circular regression,thrust,19.40,67.39,49.16,12.64
1,Linear OLS,thrust,18.39,115.11,57.03,10.83
2,Circular OLS,thrust,18.06,64.20,54.87,11.41
3,Möbius circular regression,ground,17.04,47.33,47.57,8.84
4,Circular OLS,ground,16.74,112.47,40.71,7.74
5,Circular OLS,thrust+ground,12.71,43.54,38.34,7.31
6,Linear OLS,ground,11.22,60.36,37.42,5.50
7,Linear OLS,air+ground,11.07,53.35,35.49,5.40
8,Linear OLS,air,10.94,56.30,35.65,5.63
9,Circular OLS,thrust+air,10.85,45.87,39.81,7.13


## Table 2 — Per-trajectory error distribution, all models

Median with 95% bootstrap CI (resampling trajectories), mean, interquartile range, 90th percentile and fraction of trajectories with error above 15°.

In [4]:
def dist_row(name, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Model': name, 'n': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.2f}',
            'IQR': f'{np.percentile(v, 25):.2f}–{np.percentile(v, 75):.2f}', 'Q90': f'{np.percentile(v, 90):.2f}',
            f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

per_traj_all = pd.read_csv(D + 'trajectory_level_errors_all_models.csv')
rows = [dist_row(m, g['err']) for m, g in per_traj_all.groupby('model')]
rows += [dist_row(m, v.values) for m, v in mobius_traj.items()]
table2 = pd.DataFrame(rows)
table2['_k'] = table2['Median [95% CI]'].str.split(' ').str[0].astype(float)
table2 = table2.sort_values('_k', ascending=False).drop(columns='_k').reset_index(drop=True)
save(table2, 'table2_error_distribution',
     'Distribution of per-trajectory median angular error (deg) on the held-out test set for the ANN and all baselines.',
     'error_distribution', align='lrrrrrr')

,Model,n,Median [95% CI],Mean,IQR,Q90,% > 15°
0,mobius: thrust,1101,"19.40 [18.08, 20.84]",28.14,9.58–41.65,62.60,60.0
1,linear: thrust,1101,"18.39 [16.92, 20.46]",32.51,8.32–47.80,81.74,57.2
2,circular: thrust,1101,"18.06 [16.14, 19.66]",27.50,8.11–40.37,65.18,56.0
3,mobius: ground,1101,"17.04 [14.83, 19.09]",31.61,5.91–43.72,84.60,52.7
4,circular: ground,1101,"16.74 [14.75, 19.97]",35.28,5.06–51.89,101.46,52.7
5,circular: thrust+ground,1101,"12.71 [11.43, 13.86]",22.31,5.12–31.98,53.19,44.7
6,linear: ground,1101,"11.22 [9.59, 12.81]",24.81,4.34–33.99,62.32,43.4
7,linear: air+ground,1101,"11.07 [9.46, 12.42]",23.26,4.20–32.68,56.87,42.5
8,linear: air,1101,"10.94 [9.30, 12.34]",23.15,4.41–32.46,59.11,42.7
9,circular: thrust+air,1101,"10.85 [9.89, 11.87]",22.04,5.09–28.64,56.55,40.2


## Table 3 — ANN error by flight regime

Per-trajectory median error within each regime (a trajectory contributes to every regime it visits), and the best baseline in that regime for reference.

In [5]:
def regime_row(label, v):
    v = np.asarray(v); m, lo, hi = bootstrap_ci(v)
    return {'Regime': label, 'n trajectories': len(v), 'Median [95% CI]': ci_str(m, lo, hi), 'Mean': f'{v.mean():.2f}',
            'IQR': f'{np.percentile(v, 25):.2f}–{np.percentile(v, 75):.2f}', f'% > {THRESH:g}°': f'{100 * (v > THRESH).mean():.1f}'}

rows = [regime_row('Overall', ann_pooled.values)]
rows += [regime_row(r.capitalize(), ann_by_reg.loc[ann_by_reg['regime'] == r, 'err']) for r in REGIMES]
table3 = pd.DataFrame(rows)
nb_base = t1[t1['family'] != 'ANN']
table3['Best baseline (median)'] = [f"{nb_base[c].min():.2f}  ({nb_base.loc[nb_base[c].idxmin(), 'model']})"
                                    for c in ['overall'] + REGIMES]
save(table3, 'table3_ann_by_regime',
     'ANN per-trajectory angular error (deg) on the held-out test set, overall and by flight regime, with the best baseline median in each regime for comparison.',
     'ann_regime', align='lrrrrrl')

,Regime,n trajectories,Median [95% CI],Mean,IQR,% > 15°,Best baseline (median)
0,Overall,1099,"5.35 [4.95, 5.74]",18.00,2.71–13.70,23.9,8.52 (mobius: air)
1,Upwind,261,"11.84 [8.22, 17.24]",59.73,3.55–148.22,44.8,31.97 (mobius: air)
2,Crosswind,627,"21.93 [17.23, 26.13]",40.57,6.91–62.67,57.1,26.42 (mobius: air)
3,Downwind,891,"4.85 [4.44, 5.25]",14.46,2.52–11.02,19.8,5.14 (linear: thrust+ground)


## Table 4 — Leave-one-wind-speed-out

ANN: trained on two wind speeds and tested on the third (read from the saved results). The three baselines are refit here under the same leave-one-out design: the best pooled linear and circular OLS baselines and the airspeed Möbius regression. Baseline errors are medians over all frames of the held-out trajectories; ANN errors use the trajectories after the minimum-length filter, so counts can differ slightly.

In [6]:
lowo = pd.read_csv(D + 'leave_one_wind_out_results.csv')
BASELINES = [('Linear (thrust+air)', ('thrust', 'air'), 'linear'),
             ('Circular (thrust+air+ground)', ('thrust', 'air', 'ground'), 'circular'),
             ('Möbius (air)', ('air',), 'mobius')]
rows = []
for ws in sorted(data['windspeed'].unique()):
    trn, tst = data[data['windspeed'] != ws], data[data['windspeed'] == ws].reset_index(drop=True)
    res = {}
    for label, combo, kind in BASELINES:
        ang = mobius_angle(trn, tst, combo[0]) if kind == 'mobius' else ols_angle(trn, tst, combo, kind)
        s, _, _ = summarize(frame_errors(tst, ang)); res[label] = s
    for reg in ['overall'] + REGIMES:
        a = lowo[(np.isclose(lowo['held_out_windspeed'], ws)) & (lowo['regime'] == reg)].iloc[0]
        rows.append({'Held-out wind speed (m/s)': ws, 'Regime': reg.capitalize(), 'n (ANN)': int(a['n_trajectories']),
                     'ANN [95% CI]': ci_str(a['median'], a['ci95_lo'], a['ci95_hi']),
                     **{label: f'{res[label][reg]:.2f}' for label, _, _ in BASELINES}})
table4 = pd.DataFrame(rows)
save(table4, 'table4_leave_one_wind_out',
     'Leave-one-wind-speed-out median per-trajectory angular error (deg). The ANN and the three baselines are trained on two wind speeds and tested on the held-out third.',
     'lowo', align='lllrrrr')

,Held-out wind speed (m/s),Regime,n (ANN),ANN [95% CI],Linear (thrust+air),Circular (thrust+air+ground),Möbius (air)
0,0.3,Overall,1265,"5.52 [5.18, 6.15]",17.97,13.60,9.23
1,0.3,Upwind,420,"9.07 [6.60, 13.02]",68.37,22.21,21.33
2,0.3,Crosswind,782,"15.98 [13.24, 20.57]",36.37,36.49,19.42
3,0.3,Downwind,922,"5.64 [5.23, 6.15]",6.41,7.60,7.67
4,0.4,Overall,1062,"5.05 [4.60, 5.57]",10.14,10.37,8.70
5,0.4,Upwind,251,"6.17 [4.83, 10.98]",56.16,29.70,29.60
6,0.4,Crosswind,603,"16.52 [14.30, 19.63]",33.54,33.97,25.43
7,0.4,Downwind,863,"5.28 [4.59, 5.82]",5.32,6.01,6.22
8,0.6,Overall,999,"6.93 [6.46, 7.52]",6.99,9.35,6.91
9,0.6,Upwind,160,"124.72 [99.93, 137.93]",103.47,106.31,110.06


## Table 5 — Robustness across training seeds

In [7]:
seeds = pd.read_csv(D + 'ann_multiseed_results.csv')
t5 = pd.DataFrame({'Seed': seeds['seed'], 'Median error (deg)': seeds['median_error'].map('{:.2f}'.format),
                   'Mean error (deg)': seeds['mean_error'].map('{:.2f}'.format)})
summary = pd.DataFrame([{'Seed': 'mean ± sd', 'Median error (deg)': f"{seeds['median_error'].mean():.2f} ± {seeds['median_error'].std():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].mean():.2f} ± {seeds['mean_error'].std():.2f}"},
                        {'Seed': 'range', 'Median error (deg)': f"{seeds['median_error'].min():.2f}–{seeds['median_error'].max():.2f}",
                         'Mean error (deg)': f"{seeds['mean_error'].min():.2f}–{seeds['mean_error'].max():.2f}"}])
table5 = pd.concat([t5, summary], ignore_index=True)
save(table5, 'table5_multiseed', 'ANN performance across five independently seeded training runs on the held-out test set.',
     'multiseed', align='lrr')

,Seed,Median error (deg),Mean error (deg)
0,42,5.35,18.00
1,7,5.16,17.84
2,13,5.30,17.46
3,99,5.55,18.27
4,123,5.57,18.05
5,mean ± sd,5.39 ± 0.18,17.92 ± 0.30
6,range,5.16–5.57,17.46–18.27


## Table 6 — Architecture and loss selection (3-fold trajectory-grouped CV)

Out-of-fold median per-trajectory error with 95% bootstrap CI and IQR, for (A) the initial architecture sweep with MSE loss, (B) the loss/weighting comparison at 2×15, and (C) the architecture check under the winning loss.

In [8]:
def cv_rows(df, label_fn, panel):
    return [{'Panel': panel, 'Configuration': label_fn(r), 'n': int(r['n_trajectories']),
             'CV median [95% CI]': ci_str(r['cv_median_error'], r['cv_ci95_lo'], r['cv_ci95_hi']),
             'IQR': f"{r['cv_q25']:.2f}–{r['cv_q75']:.2f}"} for _, r in df.iterrows()]

arch = lambda r: f"{int(r['layers'])} × {int(r['neurons'])}"
sweep = pd.read_csv(D + 'hparam_sweep_cv_results.csv')
loss  = pd.read_csv(D + 'loss_experiment_cv_results.csv')
comb  = pd.read_csv(D + 'combined_check_cv_results.csv')
table6 = pd.DataFrame(cv_rows(sweep, arch, 'A: architecture (MSE)')
                      + cv_rows(loss, lambda r: r['condition'].replace('_', ' '), 'B: loss (2 × 15)')
                      + cv_rows(comb, arch, 'C: architecture (geodesic, length-weighted)'))
save(table6, 'table6_architecture_loss_selection',
     'Grouped cross-validation results used to select the ANN architecture and loss.', 'selection', align='llrrr')

,Panel,Configuration,n,CV median [95% CI],IQR
0,A: architecture (MSE),2 × 30,2227,"6.66 [6.36, 7.38]",3.55–17.59
1,A: architecture (MSE),2 × 10,2227,"7.23 [6.83, 7.74]",3.96–18.12
2,A: architecture (MSE),2 × 20,2227,"7.37 [7.11, 7.74]",4.02–17.75
3,A: architecture (MSE),3 × 30,2227,"7.52 [7.13, 7.90]",4.05–18.14
4,A: architecture (MSE),1 × 30,2227,"7.52 [7.13, 8.05]",3.92–18.24
5,A: architecture (MSE),3 × 20,2227,"7.63 [7.12, 8.04]",3.96–18.37
6,A: architecture (MSE),2 × 15,2227,"7.85 [7.40, 8.37]",4.20–19.09
7,A: architecture (MSE),3 × 15,2227,"7.92 [7.51, 8.32]",4.51–18.62
8,A: architecture (MSE),1 × 20,2227,"7.98 [7.55, 8.38]",4.38–18.36
9,A: architecture (MSE),1 × 15,2227,"8.23 [7.81, 8.76]",4.73–19.15


## Table 7 — Ensemble-agreement calibration

In [9]:
ens = pd.read_csv(D + 'ann_ensemble_agreement.csv')
ens['quartile'] = pd.qcut(ens['R'], 4, labels=['Q1 (lowest agreement)', 'Q2', 'Q3', 'Q4 (highest agreement)'])
table7 = ens.groupby('quartile', observed=True).agg(n=('err', 'size'), median_R=('R', 'median'),
                                                     median_err=('err', 'median'), mean_err=('err', 'mean')).reset_index()
rho = spearmanr(ens['R'], ens['err']).statistic
table7.columns = ['Agreement quartile', 'n', 'Median R', 'Median error (deg)', 'Mean error (deg)']
table7['Median R'] = table7['Median R'].map('{:.5f}'.format)
for c in ['Median error (deg)', 'Mean error (deg)']: table7[c] = table7[c].map('{:.2f}'.format)
print(f'Spearman rho (R vs error) = {rho:.3f}')
save(table7, 'table7_ensemble_calibration',
     f'Per-trajectory error by quartile of five-model ensemble mean resultant length $R$ (Spearman $\\rho={rho:.2f}$).',
     'calibration', align='lrrrr')

Spearman rho (R vs error) = -0.528


,Agreement quartile,n,Median R,Median error (deg),Mean error (deg)
0,Q1 (lowest agreement),276,0.99855,16.44,39.95
1,Q2,274,0.99951,5.99,16.46
2,Q3,274,0.99969,3.96,8.46
3,Q4 (highest agreement),275,0.99988,2.95,7.00


## Table 8 — ANN error vs. trajectory covariates

Spearman rank correlation between per-trajectory ANN error and trajectory-level kinematic summaries on the held-out set, and median error by wind speed.

In [10]:
cov = pd.read_csv(D + 'ann_error_with_covariates.csv')
labels = {'mean_slip_angle': 'Mean |slip angle| (deg)', 'max_turning_rate': 'Max turning rate (deg/frame)',
          'mean_thrust': 'Mean thrust magnitude', 'mean_groundspeed': 'Mean groundspeed (m/s)',
          'mean_airspeed': 'Mean airspeed (m/s)', 'windspeed': 'Wind speed (m/s)'}
rows = []
for c, lab in labels.items():
    r = spearmanr(cov[c], cov['err'])
    rows.append({'Covariate': lab, 'Spearman ρ': f'{r.statistic:.3f}', 'p': f'{r.pvalue:.1e}', 'n': len(cov)})
table8 = pd.DataFrame(rows).sort_values('Spearman ρ', key=lambda s: s.astype(float).abs(), ascending=False).reset_index(drop=True)
save(table8, 'table8_covariate_correlations',
     'Spearman correlation between per-trajectory ANN error and trajectory covariates on the held-out test set.',
     'covariates', align='lrrr')
ws_tab = cov.groupby('windspeed')['err'].agg(n='size', median='median', mean='mean').round(2).reset_index()
display(ws_tab)

,Covariate,Spearman ρ,p,n
0,Mean |slip angle| (deg),0.781,1.1e-226,1099
1,Max turning rate (deg/frame),0.561,3.7e-92,1099
2,Mean airspeed (m/s),-0.537,5.9e-83,1099
3,Mean groundspeed (m/s),-0.417,2.2e-47,1099
4,Mean thrust magnitude,-0.409,1.5e-45,1099
5,Wind speed (m/s),-0.016,6.1e-01,1099


,windspeed,n,median,mean
0,0.3,405,5.38,17.87
1,0.4,371,4.82,18.11
2,0.6,323,5.81,18.03
